In [1]:
import pandas as pd

In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-04-05 20:05:30--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.86, 18.239.115.4, 18.239.115.146, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.2’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-04-05 20:05:31 (1.11 GB/s) - ‘taxi_zone_lookup.csv.2’ saved [12331/12331]



In [3]:
df_zones = pd.read_csv("taxi_zone_lookup.csv")
df_zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
import os
from sqlalchemy import create_engine, text

# Database configuration - use environment variables for security
db_user = os.getenv('DB_USER', 'root')
db_password = os.getenv('DB_PASSWORD', 'root')
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'ny_taxi_db')

# Create connection URL
db_url = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(db_url)

# Test the connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("✓ Database connection successful!")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print(f"  URL: postgresql://{db_user}:***@{db_host}:{db_port}/{db_name}")
    raise

# Load data into the database
try:
    df_zones.to_sql(name='zones', con=engine, if_exists='replace', index=False)
    print(f"✓ Loaded {len(df_zones)} records into 'zones' table")
except Exception as e:
    print(f"✗ Failed to load data: {e}")
    raise

✓ Database connection successful!
✓ Loaded 265 records into 'zones' table


In [5]:
from sqlalchemy import inspect

# Check what tables exist in the database
inspector = inspect(engine)
tables = inspector.get_table_names()
print(f"Tables in database: {tables}")

if 'zones' in tables:
    # Query the zones table
    query_result = pd.read_sql("SELECT * FROM zones LIMIT 5", engine)
    print(f"\n✓ 'zones' table exists with {len(query_result)} rows shown:")
    print(query_result)
else:
    print("\n✗ 'zones' table not found. Run the cell above to load the data.")

Tables in database: ['zones']

✓ 'zones' table exists with 5 rows shown:
   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone


## Database Setup

PostgreSQL runs via `docker-compose.yaml` in this directory with the following configuration:
- **Service name**: `pgdatabase`
- **Default database**: `ny_taxi`
- **Username**: `root`
- **Password**: `root`
- **Port**: `5432`

### Starting PostgreSQL

If PostgreSQL is not running, start it from this directory:
```bash
docker-compose up -d pgdatabase
```

### Creating the ny_taxi_db database

The database `ny_taxi_db` is already created. If you need to recreate it:
```bash
docker-compose exec pgdatabase psql -U root -d ny_taxi -c "CREATE DATABASE ny_taxi_db;"
```

### Verifying the connection

Run the cell below to test the database connection and load the taxi zones data.